# 🚀 NASA C-MAPSS Pipeline with Asymmetric NASA-Aware Scoring
### DM-TCAE + CNN-Transformer + Offline TD3 + LQR + Safety Gate
**Core Bug Fix**: Previous versions evaluated the test set across *all* sliding windows, multiplying the NASA score by ~100x. This notebook strictly evaluates only the **last window of each test engine** (as per standard C-MAPSS benchmarking rules).

**New Features for NASA Metric**:
1. **AsymmetricHuberLoss**: The supervised predictor now actively penalizes over-predictions (late predictions) by 1.5x to shift the distribution slightly early, drastically lowering the NASA score.
2. **NASA-Aware RL Reward**: The TD3 agent's reward is asymmetrically shaped to prefer under-predictions.

In [1]:
# @title 🚀 One-Click Colab / Kaggle Setup (Auto-Bootstrap `src` and `data`)
import os, sys, zlib, json, base64, urllib.request, subprocess, shutil

CWD = os.getcwd()
for p in [CWD, '/content', '/content/training']:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

try:
    import optuna
except ImportError:
    print('Installing optuna for hyperparameter optimization...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'optuna'])
    import optuna

BUNDLE_B64 = """..."""

need_unpack = not os.path.exists('src') or not os.path.exists('src/preprocessing.py')
if need_unpack:
    print('Unpacking `src/` modules...')
    raw_json = zlib.decompress(base64.b64decode(BUNDLE_B64)).decode('utf-8')
    files_map = json.loads(raw_json)
    for rel_path, content in files_map.items():
        dir_name = os.path.dirname(rel_path)
        if dir_name: os.makedirs(dir_name, exist_ok=True)
        with open(rel_path, 'w', encoding='utf-8') as fp:
            fp.write(content)
    print('  Extracted', len(files_map), 'files.')

data_dir = 'data'
os.makedirs(data_dir, exist_ok=True)
base_url = 'https://raw.githubusercontent.com/hankroark/Turbofan-Engine-Degradation/master/CMAPSSData/'
all_files = [f"{p}_{ds}.txt" for ds in ['FD001', 'FD002', 'FD003', 'FD004'] for p in ['train', 'test', 'RUL']]

def download_file(url, dest):
    try:
        subprocess.run(['curl', '-sL', '-o', dest, url], timeout=15, check=True)
        if os.path.exists(dest) and os.path.getsize(dest) > 0: return True
    except: pass
    try:
        with urllib.request.urlopen(url, timeout=15) as r, open(dest, 'wb') as f: shutil.copyfileobj(r, f)
        return True
    except: return False

missing = [f for f in all_files if not os.path.exists(os.path.join(data_dir, f)) or os.path.getsize(os.path.join(data_dir, f)) == 0]
if missing:
    print('Downloading', len(missing), 'missing C-MAPSS dataset files...')
    for fname in missing:
        if download_file(base_url + fname, os.path.join(data_dir, fname)):
            print('  Fetched', fname)
else:
    print('✅ C-MAPSS data files ready.')
print('✅ Environment ready!')

✅ C-MAPSS data files ready.
✅ Environment ready!


/home/vishal/D drive/minor/my_gnome_project/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Imports and NASA-Optimized Config

In [2]:
import os, sys, time, copy, warnings
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import optuna
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

from src.dataset import CMAPSSDataModule
from src.corruption import apply_corruption
from src.models.autoencoder import DMTCAE, MaskedReconstructionLoss
from src.models.predictor import SupervisedPredictor
from src.td3.agent import TD3Agent
from src.lqr import LQRSafetyFilter
from src.safety_gate import SafetyGate
from src.evaluation import run_full_inference
from src.metrics import calculate_metrics
from src.utils import set_seed, get_device

# --- CUSTOM ASYMMETRIC LOSS TO MINIMIZE NASA SCORE ---
class AsymmetricPredictorLoss(nn.Module):
    def __init__(self, lambda_rec=0.2, delta=10.0, penalty=1.5):
        super().__init__()
        self.lambda_rec = lambda_rec
        self.penalty = penalty
        self.huber = nn.HuberLoss(delta=delta, reduction='none')
        self.rec_criterion = nn.MSELoss()

    def forward(self, y_pred, y_true, x_hat, x_clean):
        if y_true.ndim == 1: y_true = y_true.unsqueeze(-1)
        h_loss = self.huber(y_pred, y_true)
        # Asymmetric penalty: multiply loss by `penalty` if predicting late (y_pred > y_true)
        weight = torch.where(y_pred > y_true, self.penalty, 1.0)
        main_loss = (h_loss * weight).mean()
        rec_loss = self.rec_criterion(x_hat, x_clean)
        return main_loss + self.lambda_rec * rec_loss, main_loss, rec_loss

# --- CONFIGURATION ---
SEED = 42
ALL_DATASETS = ['FD001', 'FD002', 'FD003', 'FD004']

WINDOW_LENGTH = 40
MAX_RUL = 125.0

N_OPTUNA_TRIALS = 20
EPOCHS_B_HPO = 15
EPOCHS_B_FINAL = 50
EPOCHS_A = 15
TD3_STEPS = 1500
MAX_ACTION_RL = 12.0

set_seed(SEED)
DEVICE = get_device()
print(f"Using Device: {DEVICE}")

Using Device: cuda


### 1.1 NASA C-MAPSS Scoring Function (Prognostics Metric)

The asymmetric scoring metric used in the NASA C-MAPSS benchmark is defined as:

$$d_i = \hat{y}_i - y_i = \hat{\text{RUL}}_i - \text{RUL}_i$$

$$s_i = \begin{cases} 
e^{-d_i / 13} - 1, & d_i < 0 \quad (\text{Underprediction / Early}) \\[6pt]
e^{d_i / 10} - 1, & d_i \ge 0 \quad (\text{Overprediction / Late}) 
\end{cases}$$

The total NASA score is the sum over all $N$ evaluated engines (evaluated strictly on the final window per test unit):
$$S = \sum_{i=1}^{N} s_i$$


In [3]:
def compute_nasa_score(y_pred, y_true, verbose=True):
    """
    Calculates the NASA C-MAPSS Prognostics Scoring Function.
    
    Formula:
        d_i = y_pred_i - y_true_i
        s_i = exp(-d_i / 13) - 1   if d_i < 0  (underprediction / early)
        s_i = exp(d_i / 10) - 1    if d_i >= 0 (overprediction / late)
        Total Score S = sum(s_i)
        
    Parameters:
        y_pred  : array-like of predicted RUL values
        y_true  : array-like of actual ground-truth RUL values
        verbose : bool, whether to print a breakdown of early vs late errors
        
    Returns:
        total_score : float, sum of all penalties S
        per_engine_scores : np.ndarray, individual penalties s_i
    """
    y_p = np.asarray(y_pred, dtype=np.float64).flatten()
    y_t = np.asarray(y_true, dtype=np.float64).flatten()
    assert len(y_p) == len(y_t), f"Length mismatch: {len(y_p)} predictions vs {len(y_t)} targets"
    
    # Prediction error: d = y_pred - y_true
    d = y_p - y_t
    
    # Asymmetric exponential penalty
    s = np.where(d < 0, np.exp(-d / 13.0) - 1.0, np.exp(d / 10.0) - 1.0)
    total_score = float(np.sum(s))
    
    if verbose:
        under = d < 0
        over = d >= 0
        print("=" * 55)
        print(f"📊 NASA C-MAPSS SCORE SUMMARY (N = {len(d)} engines)")
        print("=" * 55)
        print(f"  Total NASA Score        : {total_score:.2f}")
        print(f"  Early Predictions (d < 0): {np.sum(under):3d} units | Sub-score: {np.sum(s[under]):.2f}")
        print(f"  Late Predictions  (d ≥ 0): {np.sum(over):3d} units | Sub-score: {np.sum(s[over]):.2f}")
        print(f"  Max Single Penalty      : {np.max(s):.2f}")
        print("=" * 55)
        
    return total_score, s

# --- Verification & Example Demo ---
demo_true = np.array([50, 50, 50, 50])
demo_pred = np.array([40, 45, 55, 60])  # d = [-10, -5, +5, +10]

total_score, ind_scores = compute_nasa_score(demo_pred, demo_true, verbose=True)

print("\nDetailed Per-Engine Penalty Breakdown:")
for t, p, err, penalty in zip(demo_true, demo_pred, demo_pred - demo_true, ind_scores):
    kind = "Underprediction (Early)" if err < 0 else "Overprediction (Late)"
    print(f"  Actual: {t:2d} | Pred: {p:2d} | Error (d): {err:+3d} | Penalty s_i: {penalty:6.3f} [{kind}]")


📊 NASA C-MAPSS SCORE SUMMARY (N = 4 engines)
  Total NASA Score        : 3.99
  Early Predictions (d < 0):   2 units | Sub-score: 1.63
  Late Predictions  (d ≥ 0):   2 units | Sub-score: 2.37
  Max Single Penalty      : 1.72

Detailed Per-Engine Penalty Breakdown:
  Actual: 50 | Pred: 40 | Error (d): -10 | Penalty s_i:  1.158 [Underprediction (Early)]
  Actual: 50 | Pred: 45 | Error (d):  -5 | Penalty s_i:  0.469 [Underprediction (Early)]
  Actual: 50 | Pred: 55 | Error (d):  +5 | Penalty s_i:  0.649 [Overprediction (Late)]
  Actual: 50 | Pred: 60 | Error (d): +10 | Penalty s_i:  1.718 [Overprediction (Late)]


## 2. Training Loop with Asymmetric Optimization & Correct Test Extraction

In [3]:
final_results = {}

def print_optuna_callback(study, trial):
    best = study.best_value
    print(f"  [Trial {trial.number+1:02d}/{N_OPTUNA_TRIALS}] Val RMSE: {trial.value:.3f} | Best so far: {best:.3f} | Params: {trial.params}")

optuna.logging.set_verbosity(optuna.logging.WARNING)

for dataset_name in ALL_DATASETS:
    print("\n" + "="*75)
    print(f"🚀 OPTIMIZING: {dataset_name}")
    print("="*75)
    
    # 1. Load Data
    dm = CMAPSSDataModule(data_dir='data', dataset_name=dataset_name, window_length=WINDOW_LENGTH, seed=SEED, max_rul=MAX_RUL)
    dm.load_and_preprocess(verbose=False)
    train_loader, val_loader, test_loader = dm.get_loaders(batch_size=64)
    print(f"Data Loaded -> Train Windows: {len(dm.train_dataset)}, Val Windows: {len(dm.val_dataset)}")
    
    # 2. Stage A: DM-TCAE Pretraining
    print("\n--- STAGE A: DM-TCAE Pretraining ---")
    autoencoder = DMTCAE(14, 16).to(DEVICE)
    opt_ae = torch.optim.AdamW(autoencoder.parameters(), lr=1e-3, weight_decay=1e-4)
    crit_ae = MaskedReconstructionLoss()
    
    for ep in range(EPOCHS_A):
        autoencoder.train()
        losses = []
        for x, y, _, _, _ in train_loader:
            x = x.to(DEVICE)
            cx, mk = apply_corruption(x, mode='dynamic_mixed')
            rec, _ = autoencoder(cx)
            loss = crit_ae(rec, x, mk)
            opt_ae.zero_grad(); loss.backward(); opt_ae.step()
            losses.append(loss.item())
        if (ep+1) % 5 == 0:
            print(f"  Epoch {ep+1}/{EPOCHS_A} | TCAE Loss: {np.mean(losses):.4f}")
            
    # 3. Stage B: Optuna HPO
    print("\n--- STAGE B: HPO Search (using Asymmetric Loss) ---")
    def objective(trial):
        lr_b     = trial.suggest_float('lr_b', 1e-4, 2e-3, log=True)
        batch_sz = trial.suggest_categorical('batch_size', [32, 64])
        dropout  = trial.suggest_float('dropout', 0.05, 0.25)
        lam_rec  = trial.suggest_float('lambda_rec', 0.05, 0.3)
        n_layers = trial.suggest_int('n_layers', 2, 4)
        n_heads  = trial.suggest_categorical('n_heads', [4, 8])
        ffn_dim  = trial.suggest_categorical('ffn_dim', [64, 128, 256])
        weight_d = trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True)
        
        t_ld, v_ld, _ = dm.get_loaders(batch_size=batch_sz)
        
        ae_clone = copy.deepcopy(autoencoder)
        model = SupervisedPredictor(
            in_channels=14, latent_channels=16, d_model=64, n_heads=n_heads, n_layers=n_layers, ffn_dim=ffn_dim,
            dropout=dropout, max_rul=MAX_RUL, pretrained_autoencoder=ae_clone
        ).to(DEVICE)
        
        opt = torch.optim.AdamW(model.parameters(), lr=lr_b, weight_decay=weight_d)
        # Inject custom asymmetric loss to shape predictions
        crit = AsymmetricPredictorLoss(lambda_rec=lam_rec, penalty=1.5)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_B_HPO)
        
        best_val = float('inf')
        patience = 0
        
        for ep in range(EPOCHS_B_HPO):
            model.train()
            for x, y, _, _, _ in t_ld:
                x, y = x.to(DEVICE), y.to(DEVICE)
                cx, _ = apply_corruption(x, mode='dynamic_mixed')
                yb, _, rec, _ = model(cx)
                loss, _, _ = crit(yb, y, rec, x)
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            sched.step()
            
            model.eval()
            errs = []
            with torch.no_grad():
                for x, y, _, _, _ in v_ld:
                    yb, _, _, _ = model(x.to(DEVICE))
                    errs.extend(((yb - y.to(DEVICE)) ** 2).cpu().numpy().flatten())
            val_rmse = np.sqrt(np.mean(errs))
            trial.report(val_rmse, ep)
            
            if val_rmse < best_val:
                best_val = val_rmse
                patience = 0
            else:
                patience += 1
                
            if patience >= 4 or trial.should_prune():
                break
        return best_val

    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=4)
    study = optuna.create_study(direction='minimize', pruner=pruner, sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, callbacks=[print_optuna_callback])
    
    bp = study.best_params
    print(f"\n🏆 Best HPO Val RMSE: {study.best_value:.3f}")
    
    # 4. Final Stage B Training
    print("\n--- STAGE B: 50-Epoch Extreme Training (Asymmetric) ---")
    t_ld, v_ld, _ = dm.get_loaders(batch_size=bp['batch_size'])
    predictor = SupervisedPredictor(
        in_channels=14, latent_channels=16, d_model=64, n_heads=bp['n_heads'], n_layers=bp['n_layers'], ffn_dim=bp['ffn_dim'],
        dropout=bp['dropout'], max_rul=MAX_RUL, pretrained_autoencoder=copy.deepcopy(autoencoder)
    ).to(DEVICE)
    
    opt_f = torch.optim.AdamW(predictor.parameters(), lr=bp['lr_b'], weight_decay=bp['weight_decay'])
    crit_f = AsymmetricPredictorLoss(lambda_rec=bp['lambda_rec'], penalty=1.5)
    sched_f = torch.optim.lr_scheduler.OneCycleLR(opt_f, max_lr=bp['lr_b'] * 1.5, steps_per_epoch=len(t_ld), epochs=EPOCHS_B_FINAL)
    
    best_final_rmse = float('inf')
    best_state = None
    for ep in range(EPOCHS_B_FINAL):
        predictor.train()
        for x, y, _, _, _ in t_ld:
            x, y = x.to(DEVICE), y.to(DEVICE)
            cx, _ = apply_corruption(x, mode='dynamic_mixed')
            yb, _, rec, _ = predictor(cx)
            loss, _, _ = crit_f(yb, y, rec, x)
            opt_f.zero_grad(); loss.backward(); opt_f.step()
            sched_f.step()
        
        predictor.eval()
        errs = []
        with torch.no_grad():
            for x, y, _, _, _ in v_ld:
                yb, _, _, _ = predictor(x.to(DEVICE))
                errs.extend(((yb - y.to(DEVICE)) ** 2).cpu().numpy().flatten())
        val_rmse = np.sqrt(np.mean(errs))
        if val_rmse < best_final_rmse:
            best_final_rmse = val_rmse
            best_state = copy.deepcopy(predictor.state_dict())
            
        if (ep+1) % 10 == 0 or (ep+1) == EPOCHS_B_FINAL:
            print(f"  Epoch {ep+1:02d}/{EPOCHS_B_FINAL} | Val RMSE: {val_rmse:.3f}")
            
    predictor.load_state_dict(best_state)
    print(f"Final Model Loaded. Best Val RMSE: {best_final_rmse:.3f}")
    
    # 5. Stage C-E: TD3 + LQR + Gate (With NASA-Aware Reward)
    print("\n--- STAGE C-E: NASA-Aware TD3 Offline Correction ---")
    td3_agent = TD3Agent(state_dim=66, action_dim=1, max_action=MAX_ACTION_RL, device=DEVICE)
    
    from src.td3.replay_buffer import ReplayBuffer
    buffer = ReplayBuffer(66, 1, max_size=80000)
    predictor.eval()
    with torch.no_grad():
        for x, y, nc, _, _ in t_ld:
            yb, h, _, _ = predictor(x.to(DEVICE))
            yb_cpu, y_cpu = yb.cpu().numpy(), y.numpy()
            s = np.hstack([h.cpu().numpy(), yb_cpu / MAX_RUL, nc.numpy()])
            err = y_cpu - yb_cpu
            a = np.clip(err, -MAX_ACTION_RL, MAX_ACTION_RL)
            
            # NASA-Aware Reward calculation
            for i in range(len(s)):
                yc = yb_cpu[i] + a[i]
                d = yc - y_cpu[i]
                # Base accuracy (MAE)
                reward = -np.abs(d)
                # Severe penalty for late predictions
                if d > 0:
                    reward -= (1.5 * d)
                # Magnitude penalty
                reward -= 0.05 * np.abs(a[i])
                
                buffer.add(s[i], a[i], reward, s[i], False)
            
    for step in range(TD3_STEPS):
        td3_agent.train_step(buffer, batch_size=128)
        
    lqr_filter = LQRSafetyFilter(Q=1.0, R=2.0, max_correction=MAX_ACTION_RL, max_rate=MAX_ACTION_RL / 2.0)
    safety_gate = SafetyGate(max_allowed_correction=MAX_ACTION_RL)
    
    val_y_base, val_a_lqr, val_y_true = [], [], []
    with torch.no_grad():
        for x, y, nc, _, _ in v_ld:
            yb, h, _, _ = predictor(x.to(DEVICE))
            s = np.hstack([h.cpu().numpy(), yb.cpu().numpy() / MAX_RUL, nc.numpy()])
            a_td3 = td3_agent.select_action(s, noise=0.0)
            for i in range(len(yb)):
                val_y_base.append(yb[i].item())
                val_a_lqr.append(lqr_filter.filter_action(a_td3[i].item(), y_base=yb[i].item()))
                val_y_true.append(y[i].item())
    
    stats = safety_gate.calibrate(np.array(val_y_base), np.array(val_a_lqr), np.array(val_y_true))
    print(f"Gate Calibrated | Alpha: {stats['best_alpha']:.3f} | Base Val RMSE: {stats['validation_base_rmse']:.3f} -> Gated Val RMSE: {stats['validation_calibrated_rmse']:.3f}")
    
    # 6. Stage F: Final Evaluation on CLEAN TEST SET (CORRECT C-MAPSS RULES: LAST WINDOW ONLY)
    print("\n--- STAGE F: Test Set Evaluation (Last Window per Engine) ---")
    
    # ── ROOT CAUSE FIX: Extract ONLY the last window per test engine ──
    uids = dm.test_dataset.engine_ids.numpy()
    unique_uids = np.unique(uids)
    last_idx = [np.where(uids == uid)[0][-1] for uid in unique_uids]
    
    test_windows = dm.test_dataset.windows[last_idx]
    test_ruls = dm.test_dataset.ruls[last_idx]
    test_cn = dm.test_dataset.cycles_norm[last_idx]
    
    res = run_full_inference(
        predictor, td3_agent, lqr_filter, safety_gate, 
        test_windows, test_ruls, test_cn, 
        DEVICE, corruption_mode='clean'
    )
    m = calculate_metrics(res['y_final'], res['y_true'])
    m_base = calculate_metrics(res['y_base'], res['y_true'])
    
    print(f"🏆 {dataset_name} FINAL TEST RMSE (Baseline): {m_base['RMSE']:.3f}")
    print(f"🏆 {dataset_name} FINAL TEST RMSE (Full Sys): {m['RMSE']:.3f}")
    print(f"🏆 {dataset_name} FINAL NASA SCORE (Full Sys): {m['CMAPSS_Score']:.1f}")
    
    final_results[dataset_name] = {
        'Baseline RMSE': m_base['RMSE'],
        'Full System RMSE': m['RMSE'],
        'Full System Score': m['CMAPSS_Score']
    }


🚀 OPTIMIZING: FD001
Data Loaded -> Train Windows: 13407, Val Windows: 3324

--- STAGE A: DM-TCAE Pretraining ---
  Epoch 5/15 | TCAE Loss: 0.1515
  Epoch 10/15 | TCAE Loss: 0.1175
  Epoch 15/15 | TCAE Loss: 0.1112

--- STAGE B: HPO Search (using Asymmetric Loss) ---
  [Trial 01/20] Val RMSE: 14.985 | Best so far: 14.985 | Params: {'lr_b': 0.0003071057367777374, 'batch_size': 32, 'dropout': 0.1697316968394073, 'lambda_rec': 0.08900466011060913, 'n_layers': 2, 'n_heads': 8, 'ffn_dim': 128, 'weight_decay': 0.008123245085588688}
  [Trial 02/20] Val RMSE: 15.183 | Best so far: 14.985 | Params: {'lr_b': 0.0012106896936002161, 'batch_size': 32, 'dropout': 0.08668090197068676, 'lambda_rec': 0.12606056073988442, 'n_layers': 3, 'n_heads': 4, 'ffn_dim': 64, 'weight_decay': 0.00012562773503807024}
  [Trial 03/20] Val RMSE: 14.843 | Best so far: 14.843 | Params: {'lr_b': 0.0003920673972242139, 'batch_size': 32, 'dropout': 0.1528468876827223, 'lambda_rec': 0.19810364221551063, 'n_layers': 2, 'n_hea

## 3. Ultimate Multi-Dataset NASA-Aware Results

In [4]:
print("="*65)
print(f"{'Dataset':<10} | {'Baseline RMSE':<15} | {'Full System RMSE':<18} | {'NASA Score'}")
print("-" * 65)
avg_rmse = 0
for ds in ALL_DATASETS:
    r = final_results.get(ds, {'Baseline RMSE': 0, 'Full System RMSE': 0, 'Full System Score': 0})
    avg_rmse += r['Full System RMSE']
    print(f"{ds:<10} | {r['Baseline RMSE']:<15.3f} | {r['Full System RMSE']:<18.3f} | {r['Full System Score']:.1f}")
print("="*65)
print(f"🔥 FINAL AVERAGE RMSE ACROSS ALL DATASETS: {avg_rmse / len(ALL_DATASETS):.3f} 🔥")

Dataset    | Baseline RMSE   | Full System RMSE   | NASA Score
-----------------------------------------------------------------
FD001      | 15.039          | 16.051             | 386.4
FD002      | 25.785          | 27.488             | 9505.9
FD003      | 15.198          | 15.325             | 889.5
FD004      | 26.603          | 28.684             | 7955.0
🔥 FINAL AVERAGE RMSE ACROSS ALL DATASETS: 21.887 🔥
